# Task 3 - Fashion Occasion and Gender Classification

Using the same fashion images, develop models to predict who the item is intended for (`gender`), and what type of occasion or usage it is suitable for (`usage`).

## How I am framing the problem

The brief lets me treat gender and usage as two separate prediction targets, or as one combined class. I am going with **two separate targets**.<br>
The reason is a counting one. There are 5 gender values and 8 usage values, so a combined class would have up to 5 x 8 = 40 categories over the same ~38k images. The usage column already has classes with 1, 13 and 25 examples in it, and crossing those with gender would shatter them further into combinations with a single image or none at all. Two separate models keep as many examples per class as the data allows, and they also let me report per-target results, which is more useful for the report than one 40-way confusion matrix that is mostly zeros.

## What this notebook does

I am following the 4 step model development process from the Week 8 lab:

1. Determine the goal - the performance metric and a realistic target value
2. Set up the experiment - the train/validation split and the diagnostics I will use to spot over/under fitting
3. A default baseline model
4. Incremental changes based on what the diagnostics actually show

Everything before step 1 is exploration. I look at the CSV, the label distributions and the images themselves first, write down numbered observations from what I saw, and only then do the preprocessing and the modelling. Every modelling decision later in the notebook points back to one of those observations.

In [ ]:
#Load modules
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from PIL import Image

#Modelling - MLP, split, metrics
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import warnings; warnings.filterwarnings('ignore')

#Make the paths work no matter which folder jupyter was launched from
while not os.path.isdir('A2_FashionDataset') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')

Path('outputs').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)

train_csv = './A2_FashionDataset/FashionDataset/train/styles_train.csv'
image_dir = './A2_FashionDataset/FashionDataset/train/images_train'

#Settings used all the way through
img_height, img_width = 80, 60
batch_size = 32
epochs = 15

df_raw = pd.read_csv(train_csv)
print('Shape of the training csv is:', df_raw.shape)
df_raw.head()

# Exploratory Data Analysis

Before I touch a model I want to know what is actually in this dataset. The things that will decide the rest of the notebook are: what the columns are, how many labels are missing, whether every CSV row actually has an image sitting on disk, how the two target columns are distributed, and what the images themselves look like.

In [ ]:
df_raw.info()

In [ ]:
df_raw.dtypes

In [ ]:
#How much of each column is actually missing
print('Total rows:', len(df_raw))
df_raw.isna().sum()

In [ ]:
#The two Unnamed columns look like junk, lets see what is in the few rows that are not null
junk_cols = [c for c in df_raw.columns if c.startswith('Unnamed')]
print('Junk looking columns:', junk_cols)

for c in junk_cols:
    print('\n', c, '- non null count:', df_raw[c].notna().sum())
    print(df_raw.loc[df_raw[c].notna(), ['id', 'productDisplayName', c]].head())

In [ ]:
#Does every row in the csv actually have an image file behind it
files_on_disk = os.listdir(image_dir)
ids_on_disk = set(int(f.split('.')[0]) for f in files_on_disk if f.endswith('.jpg'))
ids_in_csv = set(df_raw['id'])

print('Number of jpg files on disk:', len(ids_on_disk))
print('Number of rows in the csv  :', len(ids_in_csv))
print('In the csv but no image     :', len(ids_in_csv - ids_on_disk))
print('On disk but not in the csv  :', len(ids_on_disk - ids_in_csv))

#Look at the rows that have no image
missing_image = df_raw[df_raw['id'].isin(ids_in_csv - ids_on_disk)]
missing_image[['id', 'gender', 'articleType', 'usage', 'productDisplayName']]

In [ ]:
#Distribution of the two targets, count and share of the dataset
for target in ['gender', 'usage']:
    counts = df_raw[target].value_counts(dropna=False)
    share = df_raw[target].value_counts(dropna=False, normalize=True) * 100
    print('\n---', target, '---')
    print(pd.DataFrame({'count': counts, 'percent': share.round(2)}))

In [ ]:
#Same thing as a chart, the imbalance is much easier to see this way
plt.figure(figsize=(13,4))

plt.subplot(1,2,1)
order_gender = df_raw['gender'].value_counts().index
sns.countplot(data=df_raw, x='gender', order=order_gender)
plt.title('gender')
plt.ylabel('Number of images')

plt.subplot(1,2,2)
order_usage = df_raw['usage'].value_counts().index
sns.countplot(data=df_raw, x='usage', order=order_usage)
plt.title('usage')
plt.ylabel('Number of images')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Looking at the images themselves

Counting labels only tells me half the story. The model sees pixels, not the CSV, so I want to look at a handful of images from every class and ask myself the honest question - could I personally tell these classes apart from the picture alone? If I cannot, a flattened MLP almost certainly cannot either.

In [ ]:
#Show n_sample images per class so I can eyeball what separates the classes
def show_class_samples(data, target, n_sample=6):
    classes = data[target].dropna().unique()
    classes = sorted(classes, key=lambda c: -(data[target] == c).sum())

    fig, axes = plt.subplots(len(classes), n_sample, figsize=(n_sample*1.4, len(classes)*2.0))
    for row, c in enumerate(classes):
        sample = data.loc[data[target] == c, 'id'].head(n_sample).tolist()
        for col in range(n_sample):
            ax = axes[row, col]
            ax.axis('off')
            if col < len(sample):
                ax.imshow(Image.open(os.path.join(image_dir, str(sample[col]) + '.jpg')))
            if col == 0:
                ax.set_title(c + ' (n=' + str((data[target] == c).sum()) + ')', fontsize=9, loc='left')
    plt.tight_layout()
    plt.show()
    return

#Only sample rows that actually have an image on disk
df_have_image = df_raw[df_raw['id'].isin(ids_on_disk)]

show_class_samples(df_have_image, 'gender')

In [ ]:
show_class_samples(df_have_image, 'usage')

In [ ]:
#Are the images all the same size and all colour? check a sample of 200
#the folder also carries a .DS_Store, so only walk the jpgs
jpg_files = [f for f in files_on_disk if f.endswith('.jpg')]

sizes = []
modes = []
for f in jpg_files[:200]:
    im = Image.open(os.path.join(image_dir, f))
    sizes.append(im.size)
    modes.append(im.mode)

print('Image sizes (width, height):')
print(pd.Series(sizes).value_counts())
print('\nImage modes:')
print(pd.Series(modes).value_counts())


# Observations

Now I write down what the EDA above actually showed me. Everything I do in the preprocessing and modelling sections comes back to one of these.

## Observation 1

The last two columns, `Unnamed: 10` and `Unnamed: 11`, are junk. Out of 38617 rows they are null in 38596 and 38615 rows respectively, and the handful of values that are there are fragments of product names. What has happened is that a few product names contain a comma, so when the CSV was written those names spilled over into extra columns. They carry no information about gender or usage and I will drop them.

## Observation 2

The CSV and the image folder do not agree. There are 38617 rows in the CSV but only 38613 jpg files on disk, so a small number of rows describe a product whose photo is not in the dataset.<br>
This matters more than the size of the number suggests. `flow_from_dataframe` quietly skips rows whose file it cannot find, and it does so *after* I have already built my train/validation split and my class list. The number of images the model actually trains on would then not match the number of rows I think I split, and I would be reporting metrics over a set I never verified. I would rather drop those rows myself, on purpose, and see the count, than let the loader do it silently.

## Observation 3

Missing labels are not spread evenly across the columns. `gender` has none at all, while `usage` has 72 rows with no label. There are also a few missing in `baseColour`, `season` and `productDisplayName`, but those are not my targets so they do not affect me.<br>
The important consequence is that dropping unlabelled rows has to be done **per target, not once globally**. If I dropped every row with any missing label before splitting, I would throw away 72 perfectly good rows from the gender task, which has nothing missing. Each target gets its own filtered copy of the data.

## Observation 4

Both targets are severely imbalanced, and `usage` is the worse of the two.<br>
For gender, Men is 20918 of 38617 rows and the two smallest classes, Boys and Girls, are 814 and 645. For usage, Casual alone is 29641 rows - about 77% of the dataset - while Home has **1** image, Party has 13, Travel 25 and Smart Casual 55.

This immediately rules out accuracy as the headline metric. A model that ignores the image entirely and always answers "Casual" would score around 0.77 accuracy on usage, which sounds respectable and means nothing. I need a metric that gives every class the same weight regardless of how many examples it has, so **macro-F1 is my headline metric** and accuracy is only reported next to it as a contrast. The next cell works the numbers out rather than me just asserting them.

In [ ]:
#What does a model that always answers the majority class actually score
usage_labelled = df_raw.dropna(subset=['usage'])
y_usage = usage_labelled['usage']
always_casual = pd.Series(['Casual'] * len(y_usage), index=y_usage.index)

print('Always-predict-Casual on the 8-class usage target')
print('  accuracy:', round(accuracy_score(y_usage, always_casual), 4))
print('  macro-F1:', round(f1_score(y_usage, always_casual, average='macro'), 4))

#If the 4 tiny classes are unlearnable, what is the best macro-F1 still available
n_classes_usage = y_usage.nunique()
tiny = ['Home', 'Party', 'Travel', 'Smart Casual']
learnable = n_classes_usage - len(tiny)
print('\nusage classes:', n_classes_usage, '| plausibly learnable:', learnable, '| too rare:', len(tiny))
print('Best macro-F1 if every tiny class scores 0 and every other class is perfect:',
      round(learnable / n_classes_usage, 4))

## Observation 5

This is the most important thing I found in the whole task, so it gets its own observation.

The always-Casual model scores **0.769 accuracy** while scoring a macro-F1 of about 0.11. That 0.769 is the *floor* - it is what a model achieves by learning absolutely nothing.<br>
Meanwhile, if the four tiny classes cannot be learned at all, then even a model that classifies Casual, Sports, Ethnic and Formal **perfectly** gets a macro-F1 of 4/8 = **0.50**. That 0.50 is the *ceiling*.

So on this target the accuracy floor (0.769) sits **above** the macro-F1 ceiling (0.50). The two numbers are on completely different scales and any sentence like "the model got 80%, which is better than 0.5" would be comparing things that cannot be compared. This is exactly why I have to fix macro-F1 as the headline metric before I train anything - if I set my target value using accuracy I would declare victory for a model that has learned nothing at all. It also means a usage macro-F1 in the 0.3-0.4 region is genuinely decent work, not a poor result, and I need to say so explicitly in the report so the number is not misread.

## Observation 6

Four of the usage classes are too small to learn *or* to evaluate.

Home has exactly 1 image. With an 80/20 split that image lands in either train or validation, never both, so the class is either never seen during training or never scored during validation. Party (13), Travel (25) and Smart Casual (55) are not quite as extreme but with a 20% split they contribute roughly 3, 5 and 11 validation images each - so a single image changing its prediction swings that class's F1 by a big margin, and the macro average inherits all of that noise.

The one-image class has a second, purely mechanical consequence: `train_test_split(..., stratify=...)` refuses to run when a class has fewer than 2 members. So the 8-class usage target physically cannot be stratified and I have to use a plain random split for it.

This suggests a **hypothesis** worth testing later: folding those four classes into a single 'Other' class should give a more meaningful macro-F1, because the average is no longer dominated by classes that are pure noise. I am deliberately not treating that as settled here. I will train the 8-class version first, look at its per-class results, and only then decide whether the merge is justified by evidence rather than by my assumption.

## Observation 7

The sample image grids tell me something the label counts cannot, and it is about how *learnable* the gender classes are.

Looking down the Boys and Girls rows, they are not a visually distinct kind of product. They are ordinary t-shirts, shoes and jeans that happen to be manufactured in smaller sizes. The thing that actually separates a boys t-shirt from a mens t-shirt is physical size, and size is precisely the information a product photo destroys - every item is shot alone on a white background and then cropped to the same 60x80 frame, with nothing in shot to give scale. So I expect Boys and Girls to be confused with Men and Women, and I predict that confusion will show up in the confusion matrix as a vertical smear into the two adult columns rather than as random error.

Unisex is a different problem again. Scanning that row, the items are not visually a third category between Men and Women - they are ordinary items that the retailer decided to market to everybody. Unisex is a *merchandising decision*, not a property of the pixels. Two identical-looking bags can be labelled Men and Unisex depending on business intent, which means there is genuine label noise here that no amount of model capacity can resolve.

Both of these are hypotheses at this stage. The confusion matrices in the modelling section are where I check whether the errors actually fall where I have predicted they will.

## Observation 8

The images are nearly but not perfectly uniform. Most are 60 wide by 80 tall in RGB mode, but a small proportion come back with PIL mode `L`, which means greyscale with one channel instead of three, and a few have a different size.

A network with a fixed `Flatten(input_shape=(80, 60, 3))` input cannot accept a one-channel or differently-sized array, so this has to be dealt with before the pixels reach the model. I do not need to write that code myself - `flow_from_dataframe` takes a `target_size` and defaults to `color_mode='rgb'`, so it resizes every image to 80x60 and promotes greyscale to three channels as it loads. I am noting it here so it is a decision I made knowingly rather than something that happened to work.

# Preprocessing

Each step here exists because of one of the observations above, so I have noted which one next to each.

1. Drop the `Unnamed:` columns - Observation 1, they are comma spillover and carry nothing
2. Drop CSV rows with no image file on disk - Observation 2, so the loader cannot drop them behind my back
3. Build an `id_path` column of `<id>.jpg`, which is what `flow_from_dataframe` needs as its `x_col`
4. Drop rows with no label **for the target being modelled**, per target - Observation 3, since dropping globally would cost the gender task 72 rows for no reason

Rescaling the pixels from 0-255 to 0-1 is not done here. It happens inside `ImageDataGenerator(rescale=1./255)` at load time, the same way as the Week 8 lab, so the images are never all held in memory at once.

In [ ]:
df = df_raw.copy()

#Observation 1 - drop the comma spillover columns
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

#Observation 2 - keep only rows whose image is really on disk
before = len(df)
df = df[df['id'].isin(ids_on_disk)].reset_index(drop=True)
print('Dropped', before - len(df), 'rows that had no image file')

#File names on disk are the id plus .jpg
df['id_path'] = df['id'].astype(str) + '.jpg'

print('Rows going forward:', len(df))
df.head()

In [ ]:
#Observation 3 - drop unlabelled rows for this target only, not across every column
def load_target(target):
    data = df.dropna(subset=[target]).reset_index(drop=True)
    #flow_from_dataframe insists the label column is a string
    data[target] = data[target].astype('str')
    print(target, '- rows kept:', len(data), 'of', len(df))
    return data

data_gender = load_target('gender')
data_usage = load_target('usage')

# Step 1 - Determine the goal

## The performance metric

**Macro-F1**, for the reasons set out in Observations 4 and 5. Macro-F1 averages the per-class F1 scores with equal weight, so Girls with 645 images counts exactly as much as Men with 20918. Accuracy weights each *image* equally instead, which on this data means it mostly measures how often the model says "Men" or "Casual". I will still print accuracy beside macro-F1 in every result, but only so the gap between the two is visible - the gap is itself evidence of how much the majority class is carrying.

This is where my task differs from the Week 8 lab. The lab could justify plain accuracy because CIFAR-10 has no label imbalance at all. I measured my distributions and found the opposite, so the same reasoning leads me to a different metric.

## The target value

The lab argues its target down from what others have achieved, and I will do the same. Published work on this fashion dataset reaches high accuracy on product attributes, but I have to discount that heavily for my own setup:

- I am using a **plain MLP on flattened pixels**, not a CNN. Flattening 80x60x3 into a 14400-long vector throws away every spatial relationship, so the model cannot represent "sleeve" or "collar" as a shape - it only sees which pixel positions tend to be dark. The Week 8 lab makes exactly this discount for CIFAR.
- **No data augmentation**, so no extra robustness to position or colour shifts.
- **Severe class imbalance**, which pushes the model towards the majority class every time.
- For gender specifically, Observation 7 says part of the label is not recoverable from the image at all.

Weighing all that up, my targets are:

- **gender: macro-F1 around 0.45.** Five classes, so random guessing is about 0.20 and the always-Men baseline will be near 0.14. Anything meaningfully above 0.40 means the model has learned real visual structure for at least the two big classes.
- **usage (8 class): macro-F1 around 0.30.** Remembering from Observation 5 that the ceiling here is about 0.50 once the four tiny classes are written off, 0.30 represents getting a genuine majority of the achievable signal.

These are deliberately modest and I would rather set them honestly and meet them than set them high and explain away the miss.

# Step 2 - Setup the experiment

An 80/20 train/validation split with `random_state=42` so every run in this notebook is comparable. There is a separate held-out test set supplied with the assignment, so I am not carving a third split out of the training data.

**Stratification is decided per target, and not by preference:**

- `gender` **is** stratified. Every class has hundreds of rows, and with Girls at 645 an unstratified split could easily hand train and validation noticeably different class mixes.
- `usage` (8 class) **cannot** be stratified. Home has one row and `train_test_split` raises `ValueError: The least populated class in y has only 1 member`. This is Observation 6 showing up as a hard constraint, so a plain random split is the only option.
- the merged 5-class `usage` **is** stratified, because merging removes the one-member class.

The other trap I need to design around is class indexing. `flow_from_dataframe` builds its class-to-index mapping from whatever labels appear in the dataframe it is handed. If a rare class lands only in the training slice - which Observation 6 says will happen with Home - then the train and validation generators number the classes *differently*, and every prediction would be scored against the wrong label without raising a single error. To prevent this I build one class list from the union of the train and validation labels and pass it to both generators explicitly as `classes=`.

For the same reason the validation generator uses `shuffle=False`, so `val_generator.classes` lines up row for row with the prediction order.

`plot_learning_curve` below is the instrumentation the lab calls for in this step - loss and metric, train against validation, side by side. It is what I will read in Step 4 to decide whether the model is under or over fitting.

In [ ]:
#Loss and metric, train vs val, side by side - straight from the Week 8 lab
def plot_learning_curve(train_loss, val_loss, train_metric, val_metric, metric_name='Accuracy'):
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.plot(train_loss, 'r--')
    plt.plot(val_loss, 'b--')
    plt.xlabel("epochs")
    plt.ylabel("Loss")
    plt.legend(['train', 'val'], loc='upper left')

    plt.subplot(1,2,2)
    plt.plot(train_metric, 'r--')
    plt.plot(val_metric, 'b--')
    plt.xlabel("epochs")
    plt.ylabel(metric_name)
    plt.legend(['train', 'val'], loc='upper left')

    plt.show()
    return

In [ ]:
#The one split used everywhere. stratify is passed in because usage cannot use it
def make_split(data, target, stratify=True):
    strat = data[target] if stratify else None
    train_data, val_data = train_test_split(data, test_size=0.2, random_state=42, stratify=strat)
    print('train:', len(train_data), ' val:', len(val_data))
    return train_data, val_data

#Build the train/val generator pair for an already split target
def make_generators(train_data, val_data, target):
    #One fixed class list from train and val together. without this a class that
    #only lands in train (Home) makes the two generators number the classes
    #differently and every val prediction is silently scored against the wrong label
    class_list = sorted(pd.concat([train_data[target], val_data[target]]).unique())

    train_datagen = ImageDataGenerator(rescale=1./255)
    val_datagen = ImageDataGenerator(rescale=1./255)

    train_generator = train_datagen.flow_from_dataframe(
        dataframe=train_data, directory=image_dir, x_col='id_path', y_col=target,
        classes=class_list, target_size=(img_height, img_width),
        batch_size=batch_size, class_mode='categorical')

    #shuffle=False so val_generator.classes lines up with the prediction order
    val_generator = val_datagen.flow_from_dataframe(
        dataframe=val_data, directory=image_dir, x_col='id_path', y_col=target,
        classes=class_list, target_size=(img_height, img_width),
        batch_size=batch_size, class_mode='categorical', shuffle=False)

    return train_generator, val_generator

In [ ]:
results_rows = []

#Every model in the notebook is scored through this one function so the numbers
#are comparable. labels=range(...) keeps the report aligned when a class is
#missing from val entirely, which Observation 6 says will happen
def score_model(y_true, y_pred, class_names, target, model_name, show_matrix=True):
    print('\nClassification report -', target, '-', model_name)
    print(classification_report(y_true, y_pred, labels=range(len(class_names)),
                                target_names=class_names, zero_division=0))

    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print('accuracy: %.4f   macro-F1: %.4f' % (acc, macro_f1))

    if show_matrix:
        matx = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
        plt.figure(figsize=(8,6))
        sns.heatmap(matx, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.xlabel('Predicted'); plt.ylabel('Actual')
        plt.title('Confusion matrix - ' + target + ' - ' + model_name)
        plt.tight_layout()
        plt.savefig('outputs/confusion_%s_%s.png' % (target, model_name))
        plt.show()

    results_rows.append({'target': target, 'model': model_name,
                         'accuracy': round(acc, 4), 'macro_f1': round(macro_f1, 4)})
    return acc, macro_f1

In [ ]:
#gender can be stratified, every class has hundreds of rows
print('gender')
train_gender, val_gender = make_split(data_gender, 'gender', stratify=True)

#usage cannot - Home has 1 row and stratify needs at least 2 per class
print('\nusage')
train_usage, val_usage = make_split(data_usage, 'usage', stratify=False)

#Check how the rare classes actually fell across the usage split
pd.DataFrame({'train': train_usage['usage'].value_counts(),
              'val': val_usage['usage'].value_counts()}).fillna(0).astype(int)

## Observation 9

The table above confirms Observation 6 empirically rather than theoretically. Home appears in only one of the two splits, and Party and Travel end up with a handful of validation images each. So the four small usage classes are not a hypothetical problem - they really are split in a way that makes them impossible to score meaningfully, and this is the split the model will actually be trained and evaluated on.

# Step 3 - The baseline model

Before any neural network, I need the number that a model has to beat to have been worth building. The baseline is the majority-class predictor: ignore the image completely and always answer the most common class in the training data.

I score it with exactly the same function and the same validation set as every later model, so the comparison is honest. This is the concrete version of the argument in Observation 5 - it is where the accuracy floor stops being arithmetic and becomes a measured result.

In [ ]:
#Always answer the most common training class, ignoring the image entirely
def majority_baseline(train_data, val_data, target):
    class_names = sorted(pd.concat([train_data[target], val_data[target]]).unique())
    majority = train_data[target].value_counts().idxmax()
    print('Majority class in train:', majority)

    y_true = np.array([class_names.index(v) for v in val_data[target]])
    y_pred = np.full(len(y_true), class_names.index(majority))

    return score_model(y_true, y_pred, class_names, target, 'baseline_majority', show_matrix=False)

majority_baseline(train_gender, val_gender, 'gender')
majority_baseline(train_usage, val_usage, 'usage')

## The MLP

Now I can introduce a model. I am starting from **the Week 8 lab's MLP**, unchanged in shape, because the point of step 3 is a sensible default rather than a tuned one - I want a first number that I can attribute to the data rather than to my own hyperparameter fiddling.

```
Flatten(input_shape=(80, 60, 3)) -> Dense(256, sigmoid) -> Dense(n_classes, softmax)
```

- `Flatten` turns each image into a 80 x 60 x 3 = 14400 long vector. As noted in Step 1 this is what makes it an MLP and not a CNN, and it is why my target values are modest.
- `Dense(256, sigmoid)` is the lab's hidden layer. The lab itself says 256 "is a hyper parameter and we can tune it later", which is exactly what I do in Step 4 - so I take it as given for now and let the learning curve tell me whether it is too small.
- `Dense(n_classes, softmax)` gives a probability per class. The lab's version leaves the last layer as logits and passes `from_logits=True`; I use an explicit softmax and `from_logits=False`, which the lab notes as the equivalent alternative. I prefer it because `model.predict` then returns probabilities directly and `argmax` over them is the class.
- `adam` optimiser and `categorical_crossentropy` loss, the standard pairing for multi-class from the lab.

The compiled `accuracy` metric is only there to draw the learning curve. Per Observation 4 the *judgement* is made on macro-F1, computed after training from the validation predictions.

`epochs = 15` is set at the top of the notebook. With about 30k training images on CPU each epoch is slow, so this is a compromise between letting the model converge and being able to run the notebook end to end. The learning curve will show whether 15 was enough.

In [ ]:
#The Week 8 lab MLP. hidden and dropout are arguments so the Step 4 variants
#differ from the baseline by exactly one thing and nothing else
def build_mlp(n_classes, hidden=256, activation='sigmoid', dropout=0.0):
    net = [layers.Flatten(input_shape=(img_height, img_width, 3)),
           layers.Dense(hidden, activation=activation)]
    if dropout > 0:
        net.append(layers.Dropout(dropout))
    net.append(layers.Dense(n_classes, activation='softmax'))

    model = tf.keras.Sequential(net)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

#Fit, draw the learning curve, predict on val and score
def fit_and_evaluate(model, model_name, target, train_generator, val_generator):
    class_names = list(train_generator.class_indices.keys())
    history = model.fit(train_generator, validation_data=val_generator, epochs=epochs, verbose=1)

    plot_learning_curve(history.history['loss'], history.history['val_loss'],
                        history.history['accuracy'], history.history['val_accuracy'])

    #One pass in the fixed order the val generator was built with
    val_generator.reset()
    probabilities = model.predict(val_generator, steps=len(val_generator))
    y_pred = probabilities.argmax(axis=1)
    y_true = val_generator.classes[:len(y_pred)]

    acc, macro_f1 = score_model(y_true, y_pred, class_names, target, model_name)
    model.save('models/%s_%s.keras' % (model_name, target))
    return history, acc, macro_f1

In [ ]:
#gender - the lab MLP as is
gen_train_gender, gen_val_gender = make_generators(train_gender, val_gender, 'gender')

mlp_gender = build_mlp(len(gen_train_gender.class_indices))
mlp_gender.summary()

history_gender, acc_g, f1_g = fit_and_evaluate(mlp_gender, 'mlp_default', 'gender',
                                               gen_train_gender, gen_val_gender)

## Observation 10

This is where I check the prediction I made in Observation 7 against the confusion matrix above, rather than just repeating it.

The things to read off the matrix: whether the Boys and Girls rows empty out into the Men and Women columns specifically - which is what the "size is not visible in a cropped photo" argument predicts - as opposed to scattering evenly across all five classes, which would instead mean the model simply has not learned anything for those classes. Likewise whether Unisex is being absorbed into Men, which is what I would expect if Unisex is a merchandising label rather than a visual category.

The gap between the accuracy and the macro-F1 in the report above is the other number to read. A large gap means the score is being carried by Men and Women while the three small classes contribute almost nothing, which is precisely what macro-F1 exists to expose.

In [ ]:
#usage - same MLP, 8 classes
gen_train_usage, gen_val_usage = make_generators(train_usage, val_usage, 'usage')

mlp_usage = build_mlp(len(gen_train_usage.class_indices))
history_usage, acc_u, f1_u = fit_and_evaluate(mlp_usage, 'mlp_default', 'usage',
                                              gen_train_usage, gen_val_usage)

## Observation 11

The usage report is the clearest demonstration in the notebook of why Observation 5 mattered. The accuracy sits close to the 0.769 always-Casual floor while the macro-F1 is far lower, and the per-class rows for Home, Party, Travel and Smart Casual are the reason - they will be showing precision, recall and F1 of 0.00 with support in the single digits.

Two different things are being measured there and it is worth separating them for the report. A 0.00 F1 on Home is a *measurement* problem: with one image in the entire dataset the class cannot appear in both splits, so the score is not telling me anything about the model at all. A low F1 on Formal or Ethnic, which have thousands of examples each, would be a genuine *modelling* problem. Only the second kind is something better hyperparameters could fix, and this distinction is what motivates the merged experiment further down.

# Step 4 - Incremental changes

The lab gives the decision rule, and it depends entirely on reading the learning curves rather than guessing:

**If under fitting:** increase the neurons in the hidden layer, then increase the number of hidden layers.<br>
**If over fitting:** add regularisation, add dropout, or reduce neurons/layers.

## Diagnosing the curves

So which is it? The way I read the two plots from the gender run:

- **Under fitting** looks like training and validation loss sitting close together and both still falling at epoch 15, with training accuracy plateauing at a mediocre level. The model has not got enough capacity, or has not had enough epochs, to fit even the data it can see.
- **Over fitting** looks like the two curves separating - training loss continuing down while validation loss flattens off and then turns back upward. The model is memorising the training images.

My expectation before looking is under fitting, and the reason is the input representation. A single 256-unit layer has to summarise a 14400-dimensional flattened image, and because flattening destroys spatial structure the layer is being asked to learn each pixel position independently rather than learn shapes. That is a capacity-starved setup, not a memorising one.

I am making **one change at a time** so each result attributes to a single cause. Change several things at once and a score that improves tells me nothing about which change caused it.

**Variant 1 - more neurons (512).** The lab's first response to under fitting. Everything else is held identical.

In [ ]:
#Variant 1 - 256 -> 512 neurons, nothing else touched
mlp_gender_512 = build_mlp(len(gen_train_gender.class_indices), hidden=512)
history_512, acc_512, f1_512 = fit_and_evaluate(mlp_gender_512, 'mlp_512', 'gender',
                                                gen_train_gender, gen_val_gender)

**Variant 2 - relu instead of sigmoid**, keeping the 512 neurons from Variant 1.

If Variant 1 improved things then capacity was the constraint and it is worth attacking the same problem from the other side. Sigmoid saturates: once a unit's input is strongly positive or negative its gradient goes to nearly zero and that unit effectively stops learning. On a 14400-input layer many units sit in that saturated region, so some of the capacity I just paid for is not being used. Relu does not saturate for positive inputs, so gradients keep flowing.

This is still one change from Variant 1 - only the activation differs - so the comparison stays clean.

In [ ]:
#Variant 2 - same 512 neurons, sigmoid -> relu
mlp_gender_relu = build_mlp(len(gen_train_gender.class_indices), hidden=512, activation='relu')
history_relu, acc_relu, f1_relu = fit_and_evaluate(mlp_gender_relu, 'mlp_512_relu', 'gender',
                                                   gen_train_gender, gen_val_gender)

**Variant 3 - add dropout (0.3)** on top of Variant 2.

Raising capacity twice in a row moves the model towards the over fitting side, so before settling on Variant 2 I should check whether it has crossed over. The tell in the Variant 2 curves is the training and validation loss separating, with validation flattening or rising while training keeps falling.

If that separation is there, dropout is the lab's prescribed response - it randomly zeroes 30% of the hidden units each step, so the network cannot lean on any single unit and is pushed towards features that generalise. If the curves showed no separation then this variant should not help, and a null result here is still worth recording because it is evidence that capacity rather than over fitting is the binding constraint.

In [ ]:
#Variant 3 - Variant 2 plus dropout
mlp_gender_drop = build_mlp(len(gen_train_gender.class_indices), hidden=512,
                            activation='relu', dropout=0.3)
history_drop, acc_drop, f1_drop = fit_and_evaluate(mlp_gender_drop, 'mlp_512_relu_dropout', 'gender',
                                                   gen_train_gender, gen_val_gender)

## Observation 12

Three variants is where I stop tuning the gender model. The returns from widening a flattened MLP fall away quickly, because the real limitation is not the width of the hidden layer - it is that flattening threw the spatial structure away before the first layer ever saw the image. No amount of extra units recovers a shape from a bag of pixel positions. Getting past that needs convolution, which is a different architecture rather than a hyperparameter, and it is the obvious next step if this task were carried further.

Below I apply whichever variant won on gender to the usage target as well, so both targets end up with a tuned model rather than only gender getting the benefit.

In [ ]:
#Which of the gender variants actually won on macro-F1
gender_runs = pd.DataFrame([r for r in results_rows if r['target'] == 'gender'])
gender_runs.sort_values('macro_f1', ascending=False)

In [ ]:
#Apply the tuned setup to usage as well so both targets get the same treatment
mlp_usage_tuned = build_mlp(len(gen_train_usage.class_indices), hidden=512, activation='relu')
history_usage_tuned, acc_ut, f1_ut = fit_and_evaluate(mlp_usage_tuned, 'mlp_512_relu', 'usage',
                                                      gen_train_usage, gen_val_usage)

# Testing the merged-class hypothesis

Observation 6 raised a hypothesis and Observation 11 gave it supporting evidence: four usage classes are too small to learn or to score, and their 0.00 F1 scores drag the macro average down without saying anything about the model's actual ability.

**The hypothesis:** folding Smart Casual, Travel, Party and Home into a single `Other` class gives a macro-F1 that measures what the model can really do, and lets the network spend its capacity on classes that have enough examples to learn.

I record this as a separate target name, `usage_5class`, rather than overwriting the 8-class result. Both then sit side by side in the results table and the reader can see the trade rather than just the number I preferred.

**The caveat, stated up front:** 5-class and 8-class macro-F1 are **not directly comparable**. Macro-F1 averages over however many classes there are, so changing the denominator from 8 to 5 changes the score even if the model behaves identically. Merging four hard classes into one also creates an `Other` class of about 94 images that is easier to score than the four separate classes were. So a higher number here is partly a real improvement and partly an artefact of an easier problem, and I will not claim the merged model "beat" the 8-class one on the strength of the macro-F1 alone.

In [ ]:
#Fold the four classes that Observation 6 flagged as too small into Other
mapping = {'Casual': 'Casual', 'Sports': 'Sports', 'Ethnic': 'Ethnic', 'Formal': 'Formal',
           'Smart Casual': 'Other', 'Travel': 'Other', 'Party': 'Other', 'Home': 'Other'}

data_5class = data_usage.copy()
data_5class['usage_5class'] = data_5class['usage'].map(mapping)

print(data_5class['usage_5class'].value_counts())

#The merge removed the 1 member class, so this one can be stratified
train_5, val_5 = make_split(data_5class, 'usage_5class', stratify=True)

In [ ]:
#Baseline first, the 5 class problem needs its own floor to be judged against
majority_baseline(train_5, val_5, 'usage_5class')

gen_train_5, gen_val_5 = make_generators(train_5, val_5, 'usage_5class')

mlp_5class = build_mlp(len(gen_train_5.class_indices), hidden=512, activation='relu')
history_5, acc_5, f1_5 = fit_and_evaluate(mlp_5class, 'mlp_512_relu', 'usage_5class',
                                          gen_train_5, gen_val_5)

## Observation 13

Whether the hypothesis held is decided by comparing **each model against its own baseline**, not by comparing the 5-class number against the 8-class number - which the caveat above rules out.

The fair question is: how far above its own majority baseline does each model sit? If the 5-class model beats its baseline by a wider margin than the 8-class model beats its baseline, the merge genuinely helped, because that comparison cancels out the change in the number of classes. If the margins are similar, then the merge mostly relabelled the problem rather than improving the model, and the honest thing to report is that the higher raw number is an artefact.

# Results

All runs in one table, written to `outputs/results_task3.csv`.

In [ ]:
results = pd.DataFrame(results_rows)

#Lift over that target's own baseline - the only fair way to compare across
#targets with different numbers of classes
baselines = results[results['model'] == 'baseline_majority'].set_index('target')['macro_f1']
results['baseline_macro_f1'] = results['target'].map(baselines)
results['lift_over_baseline'] = (results['macro_f1'] - results['baseline_macro_f1']).round(4)

results.to_csv('outputs/results_task3.csv', index=False)
results.sort_values(['target', 'macro_f1'], ascending=[True, False])

# Ultimate judgement

**Which model I would use, and why.**

For **gender**, I would take the best of the three tuned variants by macro-F1 from the table above. All of them beat the majority baseline by a clear margin, which is the evidence that the model has learned real visual structure rather than just discovering that Men is the most common answer. The tuning steps followed the lab's under/over fitting rule in order and each changed one thing, so I can attribute the improvement to a specific cause rather than to having tried lots of things.

For **usage**, my recommendation is the **5-class model**, but not on the basis of its higher macro-F1 - that comparison is invalid for the reason given in the caveat. I recommend it because of what Observation 11 established: four of the eight classes produce scores that are measurement noise rather than model performance. A class with one image in the entire dataset cannot be trained on and cannot be evaluated, so reporting an 8-class macro-F1 means reporting a number where half the components are meaningless. The 5-class version measures something real, and its `Other` class is at least honest about the fact that those four categories are being lumped together. The cost, which I would state in the report, is that the model can no longer distinguish Travel from Party at all - that capability is given up deliberately, in exchange for a metric that can be trusted.

**What I would not claim.** These are not strong models in absolute terms. The reason is architectural and I identified it before training anything: flattening a 80x60x3 image into a 14400-long vector destroys the spatial relationships that actually distinguish a formal shirt from a casual one, and no amount of widening the hidden layer recovers them. That is a limitation of the MLP, not of the tuning. Observation 7 adds a second ceiling that no architecture can fix - part of the gender label, particularly Boys/Girls and Unisex, is not recoverable from a fixed-size cropped product photo at all, because it encodes physical size and merchandising intent rather than appearance.

**The single most important caveat for the report:** on the 8-class usage target the accuracy of a model that has learned nothing (0.769) is higher than the macro-F1 of a model that is perfect on every learnable class (0.50). Any reader who sees "77% accuracy" and reads it as success has been misled by the metric, which is exactly why macro-F1 was fixed as the headline before a single model was trained.

**Next step if this were carried further:** a CNN. Every limitation identified above points at the same cause - the loss of spatial structure at the Flatten layer - and convolution is the direct answer to it. The class imbalance would be the second thing to attack, with class weighting so the loss stops being dominated by Casual and Men.